# LeanDojo Reprover Demo

This notebook demonstrates the use of LeanDojo reprover models:
1. Tactic generation model
2. Premise retrieval model
3. Retrieval-augmented tactic generation model

## 1. Tactic Generation Model

Generate tactics directly from proof states.

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("kaiyuy/leandojo-lean4-tacgen-byt5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("kaiyuy/leandojo-lean4-tacgen-byt5-small")

state = "n : ℕ\n⊢ gcd n n = n"
tokenized_state = tokenizer(state, return_tensors="pt")

# Generate a single tactic.
tactic_ids = model.generate(tokenized_state.input_ids, max_length=1024)
tactic = tokenizer.decode(tactic_ids[0], skip_special_tokens=True)
print(tactic, end="\n\n")

# Generate multiple tactics via beam search.
tactic_candidates_ids = model.generate(
    tokenized_state.input_ids,
    max_length=1024,
    num_beams=4,
    length_penalty=0.0,
    do_sample=False,
    num_return_sequences=4,
    early_stopping=False,
)
tactic_candidates = tokenizer.batch_decode(
    tactic_candidates_ids, skip_special_tokens=True
)
for tac in tactic_candidates:
    print(tac)

## 2. Premise Retrieval Model

Retrieve relevant premises from a corpus given a proof state.

In [2]:
import torch
from typing import Union, List
from transformers import AutoTokenizer, AutoModelForTextEncoding

tokenizer = AutoTokenizer.from_pretrained("kaiyuy/leandojo-lean4-retriever-byt5-small")
model = AutoModelForTextEncoding.from_pretrained("kaiyuy/leandojo-lean4-retriever-byt5-small")

state = "n : ℕ\n⊢ gcd n n = n"
premises = [
  "<a>vsub_eq_zero_iff_eq</a> @[simp] lemma vsub_eq_zero_iff_eq {p1 p2 : P} : p1 -ᵥ p2 = (0 : G) ↔ p1 = p2",
  "<a>is_scalar_tower.coe_to_alg_hom'</a> @[simp] lemma coe_to_alg_hom' : (to_alg_hom R S A : S → A) = algebra_map S A",
  "<a>polynomial.X_sub_C_ne_zero</a> theorem X_sub_C_ne_zero (r : R) : X - C r ≠ 0",
  "<a>forall_true_iff</a> theorem forall_true_iff : (α → true) ↔ true",
  "def <a>Nat.gcd</a> : Nat → Nat → Nat\n| 0        y := y\n| (succ x) y := have y % succ x < succ x, from mod_lt _ $ succ_pos _,\n                gcd (y % succ x) (succ x)",
  "@[simp] theorem <a>Nat.gcd_zero_left</a> (x : Nat) : gcd 0 x = x",
  "@[simp] theorem <a>Nat.gcd_succ</a> (x y : Nat) : gcd (y % succ x) y = gcd (y % succ x) (succ x)",
  "@[simp] theorem <a>Nat.mod_self</a> (n : Nat) : n % n = 0",
]  # A corpus of premises to retrieve from.

@torch.no_grad()
def encode(s: Union[str, List[str]]) -> torch.Tensor:
    """Encode texts into feature vectors."""
    if isinstance(s, str):
        s = [s]
        should_squeeze = True
    else:
        should_squeeze = False
    tokenized_s = tokenizer(s, return_tensors="pt", padding=True)
    hidden_state = model(tokenized_s.input_ids).last_hidden_state
    lens = tokenized_s.attention_mask.sum(dim=1)
    features = (hidden_state * tokenized_s.attention_mask.unsqueeze(2)).sum(dim=1) / lens.unsqueeze(1)
    if should_squeeze:
      features = features.squeeze()
    return features

@torch.no_grad()
def retrieve(state: str, premises: List[str], k: int) -> List[str]:
    """Retrieve the top-k premises given a state."""
    state_emb = encode(state)
    premise_embs = encode(premises)
    scores = (state_emb @ premise_embs.T)
    topk = scores.topk(k).indices.tolist()
    return [premises[i] for i in topk]

for p in retrieve(state, premises, k=4):
    print(p, end="\n\n")

## 3. Retrieval-Augmented Tactic Generation Model

Generate tactics using retrieved premises as context.

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("kaiyuy/leandojo-lean4-retriever-tacgen-byt5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("kaiyuy/leandojo-lean4-retriever-tacgen-byt5-small")

state = "n : ℕ\n⊢ gcd n n = n"
retrieved_premises = [
  "def <a>Nat.gcd</a> : Nat → Nat → Nat\n| 0        y := y\n| (succ x) y := have y % succ x < succ x, from mod_lt _ $ succ_pos _,\n                gcd (y % succ x) (succ x)",
  "@[simp] theorem <a>Nat.mod_self</a> (n : Nat) : n % n = 0",
]
input = "\n\n".join(retrieved_premises + [state])
print("------ INPUT ------\n", input)
tokenized_input = tokenizer(input, return_tensors="pt", max_length=2300, truncation=True)

# Generate a single tactic.
tactic_ids = model.generate(tokenized_input.input_ids, max_length=1024)
tactic = tokenizer.decode(tactic_ids[0], skip_special_tokens=True)
print("\n------ OUTPUT ------")
print(tactic, end="\n\n")

# Generate multiple tactics via beam search.
tactic_candidates_ids = model.generate(
    tokenized_input.input_ids,
    max_length=1024,
    num_beams=4,
    length_penalty=0.0,
    do_sample=False,
    num_return_sequences=4,
    early_stopping=False,
)
tactic_candidates = tokenizer.batch_decode(
    tactic_candidates_ids, skip_special_tokens=True
)
for tac in tactic_candidates:
    print(tac)

# Load full proof :traced

In [ ]:
import json
import json
data = json.load(open("complete_proofs.json", "rt"))


OUT_EDGES_JSONL = "tripartite_edges_all.jsonl"
OUT_THEOREMS_JSONL = "theorem_registry_all.jsonl"
OUT_PREMISES_JSONL = "premise_registry_unique.jsonl"

# Load all edges
with open(OUT_EDGES_JSONL, "r", encoding="utf-8") as f:
    edges = [json.loads(line) for line in f if line.strip()]

# Load all theorems
with open(OUT_THEOREMS_JSONL, "r", encoding="utf-8") as f:
    theorems = [json.loads(line) for line in f if line.strip()]

# Load all premises
with open(OUT_PREMISES_JSONL, "r", encoding="utf-8") as f:
    premises = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(edges)} edges")
print(f"Loaded {len(theorems)} theorems")
print(f"Loaded {len(premises)} premises")

In [76]:
theorems[0]

In [151]:
# Import and reload myutils_reprover module
import importlib
import sys

# Remove from cache if already imported
if '00_myutils_reprover' in sys.modules:
    importlib.reload(sys.modules['00_myutils_reprover'])

from myutils_reprover import (
    extract_theorem_name,
    create_theorem_from_complete_proof,
    create_theorems_from_complete_proofs,
    test_matching_success
)

print("✓ myutils_reprover imported and reloaded successfully!")

In [78]:
# Test matching on 1000 random samples
results = test_matching_success(data, edges, num_samples=200)


In [95]:
from lean_dojo import Dojo
from tqdm import tqdm

def run_proof_tactics(i, data, theorem_registry, printing=True):
    """Simple version: just run tactics without premise extraction."""
    if printing:
        print(f"Running tactics for proof index: {i}")
    
    # Use theorem_registry for theorem matching
    theorem = create_theorem_from_complete_proof(data[i], theorem_registry)
    
    if theorem is None:
        if printing:
            print("Could not map proof to a Mathlib theorem via theorem_registry.")
        return False
    
    if printing:
        print(f"Theorem: {theorem}")
        for elem in data[i]:
            print(elem)

    try:
        dojo, state = Dojo(theorem).__enter__()
    except Exception as e:
        if printing:
            print(f"Error creating Dojo for theorem {theorem}: {e}")
        return False  # Failure

    tactics_text = data[i][1]
    tactics = [line.strip() for line in tactics_text.splitlines() if line.strip()]

    # Skip first tactic if it is just "by"
    if tactics and tactics[0] == "by":
        tactics = tactics[1:]

    cur_state = state
    success = True
    for idx, tac in enumerate(tactics):
        if printing:
            print(f">>> Applying tactic: {tac}")
        try:
            cur_state = dojo.run_tac(cur_state, tac)
            if printing:
                print(cur_state)
        except Exception as e:
            if printing:
                print(f"Error running tactic '{tac}': {e}")
            success = False
            break

    return success

In [96]:
# Run for i from 1 to 100 (inclusive) with tqdm progress bar and print progress, % success, proof finished states
success_count = 0
finished_count = 0
total = 100
results = []

with tqdm(total=total, desc="Proof Progress", unit="proof", dynamic_ncols=True, leave=True) as pbar:
    for i in range(1, total + 1):
        success = run_proof_tactics(i, data, theorem_registry, printing=False)
        results.append(success)
        finished_count += 1
        if success:
            success_count += 1
        percent_done = 100 * finished_count / total
        percent_success = 100 * success_count / finished_count
        pbar.set_postfix({
            'Success Rate': f"{percent_success:.1f}%",
            'Finished': f"{finished_count}/{total}"
        }, refresh=True)
        pbar.update(1)

In [98]:
for i in range(10,12):
    success = run_proof_tactics(i, data, edges, printing=1)

In [92]:
## attempting Reprover to succesfully get tactics

# attempting Reprover to succesfully get tactics

In [113]:
from lean_dojo import Dojo
from tqdm import tqdm

i=1
theorem = create_theorem_from_complete_proof(data[i], theorem_registry)
dojo, state = Dojo(theorem).__enter__()

tokenized_state = tokenizer(state.pp, return_tensors="pt")

# Generate a single tactic.
tactic_ids = model.generate(tokenized_state.input_ids, max_length=1024)
tactic = tokenizer.decode(tactic_ids[0], skip_special_tokens=True)
print(tactic, end="\n\n")
n_tac=64
# Generate multiple tactics via beam search.
tactic_candidates_ids = model.generate(
    tokenized_state.input_ids,
    max_length=1024,
    num_beams=n_tac,
    length_penalty=0.0,
    do_sample=False,
    num_return_sequences=n_tac,
    early_stopping=False,
)
tactic_candidates = tokenizer.batch_decode(
    tactic_candidates_ids, skip_special_tokens=True
)



In [114]:
for tac in tactic_candidates:
    print(tac)
    try:
        cur_state = dojo.run_tac(state, tac)
    except Exception as e:
        if "LeanError(error='internal exception #4')" in str(e):
            pass

In [112]:
run_proof_tactics(i, data, edges, printing=1)

In [ ]:

# # state = "n : ℕ\n⊢ gcd n n = n"
# # state=edges[0]['state_before']


# # Generate multiple tactics via beam search.
# tactic_candidates_ids = model.generate(
#     tokenized_state.input_ids,
#     max_length=1024,
#     num_beams=4,
#     length_penalty=0.0,
#     do_sample=False,
#     num_return_sequences=4,
#     early_stopping=False,
# )
# tactic_candidates = tokenizer.batch_decode(
#     tactic_candidates_ids, skip_special_tokens=True
# )
# for tac in tactic_candidates:
#     print(tac)

# getting lemmas


## running proof with tactics and resolving at proof time? why are we doing this

In [89]:

# Run proof tactics with the new signature
result = run_proof_tactics(
    i=0,  # proof index
    data=data,
    theorem_registry=theorem_registry,  # Use theorem_registry instead of edges
    premise_registry=premise_registry,  # Use premise_registry instead of edges
    printing=True,
    print_states=False,
    show_new_lemmas_per_step=True
)

# Access results
print(f"Success: {result['success']}")
print(f"Theorem: {result['theorem_full_name']}")
print(f"Resolved lemmas: {len(result['resolved_best'])}")
print(f"Unresolved: {len(result['unresolved'])}")

In [62]:

# Run proof tactics for a specific proof
result = run_proof_tactics(
    i=0,  # proof index
    data=data,
    edges=edges,
    printing=True,
    print_states=False,
    show_new_lemmas_per_step=True
)

# Access results
print(f"Success: {result['success']}")
print(f"Theorem: {result['theorem_full_name']}")
print(f"Resolved lemmas: {len(result['resolved_best'])}")
print(f"Unresolved: {len(result['unresolved'])}")

In [52]:
result

# Load Algebra  

# HEURISTIC Prover

In [65]:
import json
import random
import re
import time
from collections import defaultdict
from lean_dojo import *

# ============================================================
# Progress bar (with ETA)
# ============================================================

def progress_bar(i, n, start_time, width=30, prefix=""):
    elapsed = time.time() - start_time
    rate = (i + 1) / elapsed if elapsed > 0 else 0.0
    remaining = (n - (i + 1)) / rate if rate > 0 else float("inf")
    filled = int(width * (i + 1) / n) if n > 0 else width
    bar = "█" * filled + "░" * (width - filled)
    eta_str = f"{remaining:6.1f}s" if remaining != float("inf") else "   infs"
    print(f"\r{prefix}[{bar}] {i+1}/{n} | ETA {eta_str}", end="")

# ============================================================
# Safe state helpers
# ============================================================

GOAL_MARK = "⊢"

def is_error(x):
    return isinstance(x, LeanError)

def is_finished(x):
    # LeanDojo uses ProofFinished to signal success
    return isinstance(x, ProofFinished)

def state_pp(x):
    # Only states have .pp
    return getattr(x, "pp", "")

def progressed(old_state, new_state):
    # If proof finished, that is "progress"
    if is_finished(new_state):
        return True
    return state_pp(old_state).strip() != state_pp(new_state).strip()

def extract_goal_block(state_pp_str: str):
    if GOAL_MARK not in state_pp_str:
        return ""
    idx = state_pp_str.rfind(GOAL_MARK)
    return state_pp_str[idx:].strip()

def extract_locals_with_types(state_pp_str: str):
    out = []
    for line in state_pp_str.splitlines():
        line = line.strip()
        if not line or line.startswith(GOAL_MARK):
            continue
        if " : " in line:
            name, ty = line.split(" : ", 1)
            name = name.strip()
            ty = ty.strip()
            if re.match(r"^[a-zA-Z_][a-zA-Z0-9_']*$", name):
                out.append((name, ty))
    return out

# ============================================================
# Tactic parsing / family mapping
# ============================================================

def tactic_family(tac: str):
    t = (tac or "").strip()
    if t.startswith(("simp_all",)):
        return "simp_all"
    if t.startswith(("simp_rw",)):
        return "simp_rw"
    if t.startswith(("simp",)):
        return "simp"
    if t.startswith(("rw", "rwa", "rewrite")):
        return "rw"
    if t.startswith(("intro", "intros", "rintro")):
        return "intro"
    if t.startswith(("cases", "rcases")):
        return "cases"
    if t.startswith(("ext", "ext1", "ext2")):
        return "ext"
    if t.startswith(("constructor",)):
        return "constructor"
    if t.startswith(("apply",)):
        return "apply"
    if t.startswith(("exact",)):
        return "exact"
    if t.startswith(("aesop",)):
        return "aesop"
    if t in {"rfl", "decide", "assumption"}:
        return t
    return "other"

def parse_bracket_lemmas(tac: str):
    t = tac or ""
    m = re.search(r"\[([^\]]+)\]", t)
    if not m:
        return []
    content = m.group(1)
    toks = []
    for raw in content.split(","):
        tok = raw.strip()
        if tok == "*":
            toks.append(tok)
        else:
            if re.match(r"^[A-Za-z_][A-Za-z0-9_'.]*$", tok):
                toks.append(tok)
    return toks

# ============================================================
# LeanDojo tactic execution
# (This now also prints the tactic string and state when a successful tactic is found.)

def try_tac(dojo, state, tac_str, print_on_success=False):
    res = dojo.run_tac(state, tac_str)
    if is_error(res):
        return None
    if print_on_success and progressed(state, res):
        print("\n─────────────────────────────────────────────")
        print(f"Tactic succeeded: {tac_str!r}")
        print(f"State after tactic:\n{state_pp(res)}")
        print("─────────────────────────────────────────────\n")
    return res

# ============================================================
# Family-aware execution with premises
# (Rewrite to record and print successful tactics and resulting states for reproducibility.)

def run_family(dojo, state, family, recorded_edge):
    """
    Returns:
      - (tactic, new_state) on success (actual tactic string as used in run_tac)
      - None on failure
    """
    pp0 = state_pp(state)
    locals_with_types = extract_locals_with_types(pp0)
    local_names = [n for n, _ in locals_with_types]
    goal = extract_goal_block(pp0)

    recorded_premises = recorded_edge.get("premises", []) or []
    recorded_tac = recorded_edge.get("tactic", "") or ""
    bracket_lemmas = parse_bracket_lemmas(recorded_tac)

    # Always-cheap closers
    for cheap in ["assumption", "rfl"]:
        ns = try_tac(dojo, state, cheap, print_on_success=True)
        if ns is not None and progressed(state, ns):
            return cheap, ns

    if family == "intro":
        if "⊢ ∀" in goal or "→" in goal:
            tac = "intro"
            ns = try_tac(dojo, state, tac, print_on_success=True)
            if ns is not None:
                return tac, ns
        return None

    if family == "simp":
        if "simp only" in recorded_tac and bracket_lemmas:
            lemmas_str = ", ".join(bracket_lemmas[:10])
            tac = f"simp only [{lemmas_str}]"
            ns = try_tac(dojo, state, tac, print_on_success=True)
            if ns is not None and progressed(state, ns):
                return tac, ns

        if bracket_lemmas and "simp" in recorded_tac and "only" not in recorded_tac:
            lemmas = [x for x in bracket_lemmas if x != "*"]
            if lemmas:
                tac = f"simp [{', '.join(lemmas[:10])}]"
                ns = try_tac(dojo, state, tac, print_on_success=True)
                if ns is not None and progressed(state, ns):
                    return tac, ns

        if recorded_premises:
            tac = f"simp [{', '.join(recorded_premises[:10])}]"
            ns = try_tac(dojo, state, tac, print_on_success=True)
            if ns is not None and progressed(state, ns):
                return tac, ns

        useful = [n for n in local_names if n.startswith("h") or n.startswith("ih")]
        if useful:
            tac = f"simp [{', '.join(useful[:10])}]"
            ns = try_tac(dojo, state, tac, print_on_success=True)
            if ns is not None and progressed(state, ns):
                return tac, ns

        for t in ["simp", "simp_all"]:
            ns = try_tac(dojo, state, t, print_on_success=True)
            if ns is not None and progressed(state, ns):
                return t, ns

        return None

    if family == "simp_all":
        tac = "simp_all"
        ns = try_tac(dojo, state, tac, print_on_success=True)
        if ns is not None and progressed(state, ns):
            return tac, ns
        inner_result = run_family(dojo, state, "simp", recorded_edge)
        if inner_result is not None:
            return inner_result
        return None

    if family == "simp_rw":
        candidates = []
        candidates.extend([x for x in bracket_lemmas if x != "*"])
        candidates.extend(recorded_premises)
        candidates.extend([n for n in local_names if n.startswith("h") or n.startswith("ih")])

        for lem in candidates[:12]:
            tac = f"simp_rw [{lem}]"
            ns = try_tac(dojo, state, tac, print_on_success=True)
            if ns is not None:
                return tac, ns
        return None

    if family == "rw":
        recorded_list = [x for x in bracket_lemmas if x != "*"]

        if recorded_list:
            tac = f"rw [{', '.join(recorded_list[:8])}]"
            ns = try_tac(dojo, state, tac, print_on_success=True)
            if ns is not None:
                return tac, ns

        candidates = []
        candidates.extend(recorded_list)
        candidates.extend(recorded_premises)
        candidates.extend([n for n in local_names if n.startswith("h") or n.startswith("ih")])

        for lem in candidates[:20]:
            tac = f"rw [{lem}]"
            ns = try_tac(dojo, state, tac, print_on_success=True)
            if ns is not None:
                return tac, ns

        return None

    if family == "cases":
        inductive_hints = ("Nat", "Fin", "Sum", "Prod", "Sigma", "Subtype", "Exists", "And", "Or")
        ranked = []
        for n, ty in locals_with_types:
            score = 0
            if any(h in ty for h in inductive_hints):
                score += 2
            if n in {"n", "m", "k"} or n.endswith("n") or n.endswith("m"):
                score += 1
            if n.startswith("h") or n.startswith("ih"):
                score -= 1
            ranked.append((score, n))
        ranked.sort(reverse=True)

        for _, n in ranked[:15]:
            tac = f"cases {n}"
            ns = try_tac(dojo, state, tac, print_on_success=True)
            if ns is not None:
                return tac, ns
        return None

    if family == "ext":
        for t in ["ext", "ext1", "ext2"]:
            ns = try_tac(dojo, state, t, print_on_success=True)
            if ns is not None:
                return t, ns
        return None

    if family == "constructor":
        tac = "constructor"
        ns = try_tac(dojo, state, tac, print_on_success=True)
        if ns is not None:
            return tac, ns
        return None

    if family == "exact":
        for n in local_names:
            tac = f"exact {n}"
            ns = try_tac(dojo, state, tac, print_on_success=True)
            if ns is not None:
                return tac, ns
        for lem in recorded_premises[:10]:
            tac = f"exact {lem}"
            ns = try_tac(dojo, state, tac, print_on_success=True)
            if ns is not None:
                return tac, ns
        return None

    if family == "apply":
        candidates = []
        candidates.extend([x for x in bracket_lemmas if x != "*"])
        candidates.extend(recorded_premises)

        for lem in candidates[:15]:
            tac = f"apply {lem}"
            ns = try_tac(dojo, state, tac, print_on_success=True)
            if ns is not None:
                return tac, ns
        return None

    if family == "aesop":
        tac = "aesop"
        ns = try_tac(dojo, state, tac, print_on_success=True)
        if ns is not None:
            return tac, ns
        return None

    if family in {"rfl", "decide", "assumption"}:
        tac = family
        ns = try_tac(dojo, state, tac, print_on_success=True)
        if ns is not None:
            return tac, ns
        return None

    # other: controlled high-yield bundle
    for t in ["simp", "simp_all", "aesop", "constructor", "ext", "assumption"]:
        ns = try_tac(dojo, state, t, print_on_success=True)
        if ns is not None and progressed(state, ns):
            return t, ns

    return None

# ============================================================
# Fallback bundle (cheap ATP-ish)
# (Also prints fallback tactic and state.)

FALLBACK_TACTICS = ["simp", "simp_all", "aesop", "linarith", "nlinarith", "ring"]

def fallback_bundle(dojo, state):
    for fb in FALLBACK_TACTICS:
        ns = try_tac(dojo, state, fb, print_on_success=True)
        if ns is not None and progressed(state, ns):
            return fb, ns
    return None, None


In [ ]:
# ============================================================
# Main: Try for 10 random theorems
# ============================================================
from typing import Any

import random
import sys

with open("tripartite_edges.jsonl", "r", encoding="utf-8") as f:
    edges = [json.loads(line) for line in f]

theorem_edges = defaultdict(list)
for e in edges:
    theorem_edges[e["theorem"]].append(e)

for thm in theorem_edges:
    theorem_edges[thm].sort(key=lambda x: len(x.get("state_before", "")))

all_theorems = list(theorem_edges.keys())
n_try = min(10, len(all_theorems))
random_theorems = random.sample(all_theorems, n_try)

results = []
completed_proofs = []

def progress_bar_inline(i, n, start, prefix=""):
    bar_len = 30
    progress = i / n if n > 0 else 0
    filled_len = int(bar_len * progress)
    bar = '#' * filled_len + '-' * (bar_len - filled_len)
    elapsed = time.time() - start
    msg = f"{prefix}[{bar}] {i}/{n} Elapsed: {elapsed:.1f}s"
    print("\r" + msg, end="", flush=True)
    if i == n:
        print() # newline after last bar

for thm_idx, thm_name in enumerate(random_theorems, 1):
    print("="*60)
    print(f"🔍 Theorem {thm_idx}/{n_try}: {thm_name}")

    first = theorem_edges[thm_name][0]
    file_path = first["file"].replace("\\", "/")
    theorem = create_theorem_from_edge(first)

    from lean_dojo import Dojo
    dojo, state = Dojo(theorem).__enter__()

    fams = [tactic_family(e["tactic"]) for e in theorem_edges[thm_name]]

    print(f"  Initial state ID: {state.id}")
    print(f"  Replaying {len(fams)} tactic families...")

    matches = 0
    start = time.time()
    tactic_trace = []
    proof_completed = False

    for i, (fam, edge) in enumerate(zip(fams, theorem_edges[thm_name])):
        progress_bar_inline(i, len(fams), start, prefix="  ")
        result = run_family(dojo, state, fam, edge)
        if result is not None:
            tac_str, ns = result
            matches += 1
            print(f"\nStep {i+1} succeeded with tactic: {tac_str!r}")
            print(f"State after tactic {i+1} (state.id={getattr(ns,'id','?')}):\n{state_pp(ns)}")
            tactic_trace.append((tac_str, state_pp(ns)))
            if is_finished(ns):
                print(f"\n    Step {i+1}: ✓ {fam} → PROOF COMPLETE")
                state = ns
                proof_completed = True
                break
            print(f"\n    Step {i+1}: ✓ {fam} → State {getattr(ns,'id','?')}")
            state = ns
            continue

        # Fallback
        fb_name, fb_ns = fallback_bundle(dojo, state)
        if fb_ns is not None:
            matches += 1
            print(f"\nStep {i+1} succeeded with fallback tactic: {fb_name!r}")
            print(f"State after fallback tactic (state.id={getattr(fb_ns,'id','?')}):\n{state_pp(fb_ns)}")
            tactic_trace.append((fb_name, state_pp(fb_ns)))
            if is_finished(fb_ns):
                print(f"\n    Step {i+1}: ~ {fam} failed, fallback ✓ {fb_name} → PROOF COMPLETE")
                state = fb_ns
                proof_completed = True
                break
            print(f"\n    Step {i+1}: ~ {fam} failed, fallback ✓ {fb_name} → State {getattr(fb_ns,'id','?')}")
            state = fb_ns
            continue

        print(f"\n    Step {i+1}: × {fam}")
        # Optionally still record failure for full trace
        tactic_trace.append((None, None))
        # Don't advance the state
    # Print bar closed at the end of this theorem if not already printed
    progress_bar_inline(len(fams), len(fams), start, prefix="  ")

    success_rate = matches / len(fams) if fams else 0.0
    print(f"  Result: {matches}/{len(fams)} steps succeeded ({success_rate:.1%})")
    print("\nFull reproducible tactic trace for this proof:")
    for idx, (tac, state_str) in enumerate(tactic_trace, 1):
        if tac is not None:
            print(f"Step {idx}: tactic = {repr(tac)}")
            print(f"State after step {idx}:\n{state_str}\n{'-'*40}")
        else:
            print(f"Step {idx}: tactic failed.\n{'-'*40}")

    # Store result, splitting complete/incomplete proofs:
    result_entry = {
        "theorem": thm_name,
        "matches": matches,
        "total": len(fams),
        "trace": tactic_trace,
        "success_rate": success_rate,
        "completed": proof_completed,
    }
    if proof_completed:
        completed_proofs.append(result_entry)
    else:
        results.append(result_entry)


In [ ]:

# # ============================================================
# # Main
# # ============================================================

# with open("tripartite_edges.jsonl", "r", encoding="utf-8") as f:
#     edges = [json.loads(line) for line in f]

# theorem_edges = defaultdict(list)
# for e in edges:
#     theorem_edges[e["theorem"]].append(e)

# for thm in theorem_edges:
#     theorem_edges[thm].sort(key=lambda x: len(x.get("state_before", "")))

# print(f"Found {len(theorem_edges)} unique theorems with recorded tactics")

# sample_theorems = random.sample(list(theorem_edges.keys()),
#                                 min(10, len(theorem_edges)))

# results = []

# for thm_name in sample_theorems:
#     print(f"\n🔍 Sanity checking theorem: {thm_name}")
#     first = theorem_edges[thm_name][0]
#     file_path = first["file"].replace("\\", "/")

#     try:
#         theorem = Theorem(repo, file_path, thm_name)
#         dojo, state = Dojo(theorem).__enter__()

#         fams = [tactic_family(e["tactic"]) for e in theorem_edges[thm_name]]
#         print(f"  Initial state ID: {state.id}")
#         print(f"  Replaying {len(fams)} tactic families...")

#         matches = 0
#         start = time.time()

#         for i, (fam, edge) in enumerate(zip(fams, theorem_edges[thm_name])):
#             progress_bar(i, len(fams), start, prefix="  ")

#             ns = run_family(dojo, state, fam, edge)

#             if ns is None:
#                 fb_name, fb_ns = fallback_bundle(dojo, state)
#                 if fb_ns is not None:
#                     matches += 1
#                     if is_finished(fb_ns):
#                         print(f"\n    Step {i+1}: ~ {fam} failed, fallback ✓ {fb_name} → PROOF COMPLETE")
#                         state = fb_ns
#                         break
#                     print(f"\n    Step {i+1}: ~ {fam} failed, fallback ✓ {fb_name} → State {fb_ns.id}")
#                     state = fb_ns
#                     continue

#                 print(f"\n    Step {i+1}: × {fam}")
#                 continue

#             matches += 1
#             if is_finished(ns):
#                 print(f"\n    Step {i+1}: ✓ {fam} → PROOF COMPLETE")
#                 state = ns
#                 break

#             print(f"\n    Step {i+1}: ✓ {fam} → State {ns.id}")
#             state = ns

#         success_rate = matches / len(fams) if fams else 0.0
#         print(f"  Result: {matches}/{len(fams)} steps succeeded ({success_rate:.1%})")

#         results.append(success_rate)
#         dojo.__exit__(None, None, None)

#     except Exception as e:
#         print(f"\n  FAILED: {e}")
#         results.append(0.0)

# avg = sum(results) / len(results) if results else 0.0
# print("\n📊 Summary")
# print(f"Theorems tested: {len(results)}")
# print(f"Average success rate: {avg:.1%}")

# if avg > 0.25:
#     print("✅ Better baseline: bracket-lemma rw/simp + fallbacks are working.")
# else:
#     print("⚠️ Still low: to jump further you need real premise capture (lemmas used) into the JSON.")


# DAG run for simple theorom

In [ ]:
import time

# Set this at the top so the user can easily configure the time limit:
DAG_TIME_LIMIT_SECONDS = 60.0  # Changeable timeout for DAG proof search

X = 64  # Use 64 beams for beam search

import pickle
from collections import defaultdict
from lean_dojo import LeanError

best_first = True  # Toggle: If True, only pursue the first successful child per node, else (old) up to MAX_CHILDREN_PER_NODE

MAX_CHILDREN_PER_NODE = 3  # Only pursue up to 3 successful children states per node, can be tuned, IGNORED if best_first

dag_nodes = []
dag_edges = []
dag_results = []  # Will store transitions: source, tactic, dest, etc.
dag_node_states = {}  # NEW -- map: tuple(tactic_path) -> {'pp': <pp string>, ...}

states_to_explore = [(state, [])]  # List of (state, tactic_path)

# Save root node state right away
root_pp = state.pp if hasattr(state, "pp") else str(state)
dag_node_states[tuple([])] = {
    "pp": root_pp,
    # You could serialize more info if needed
}

found_complete_proof = False
cur_depth = 0
max_depth = 100  # hard upper limit to prevent infinite loop if never proves

def is_state_proof_complete(st):
    # st: LeanDojo tactic state (or whatever is in `state`)
    # New logic: check if state is ProofFinished
    if st is None:
        return True
    # ProofFinished appears as a class (or instance) with a typical message
    if type(st).__name__ == "ProofFinished":
        return True
    # Fallbacks for various LeanDojo/Lean goals
    if hasattr(st, 'goals') and hasattr(st.goals, '__len__'):
        if len(st.goals) == 0:
            return True
    if hasattr(st, 'pp'):
        pp_str = st.pp
        if pp_str is None:
            return True
        if "no goals" in pp_str.lower() or "⊢ true" in pp_str:
            return True
        # Sometimes the string conversion of ProofFinished is in .pp
        if "ProofFinished" in pp_str:
            return True
    return False

start_time = time.time()

while not found_complete_proof and cur_depth < max_depth and len(states_to_explore) > 0:
    # Check for timeout first
    elapsed = time.time() - start_time
    if elapsed > DAG_TIME_LIMIT_SECONDS:
        print(f"Timeout reached after {elapsed:.1f} seconds (limit={DAG_TIME_LIMIT_SECONDS}). Stopping search and constructing DAG with progress so far.")
        break

    print(f"\n--- Exploring depth {cur_depth} ---")
    new_states_to_explore = []
    num_success = 0
    num_failed = 0
    node_stats_this_level = []  # [(index, success%, fail%, total)]
    for i, (cur_state, tactic_path) in enumerate(states_to_explore):
        print(f"  -> [{cur_depth}] Exploring node {i+1}/{len(states_to_explore)} with path: {tactic_path}")

        # Always save the current state's proof state in DAG node states (even before expansion/children)
        cur_pp = cur_state.pp if hasattr(cur_state, "pp") else str(cur_state)
        dag_node_states[tuple(tactic_path)] = {
            "pp": cur_pp,
        }

        tokenized_input = tokenizer(cur_state.pp, return_tensors="pt")
        # Generate top-X tactic candidates via beam search.
        tactic_candidates_ids = model.generate(
            tokenized_input.input_ids,
            max_length=1024,
            num_beams=X,
            length_penalty=0.0,
            do_sample=False,
            num_return_sequences=X,
            early_stopping=False,
        )
        tactic_candidates = tokenizer.batch_decode(
            tactic_candidates_ids, skip_special_tokens=True
        )

        print(f"    Got {len(tactic_candidates)} tactic candidates, trying each...")
        success_for_this_node = 0
        fail_for_this_node = 0
        complete_proof_for_this_node = False

        if best_first:
            first_success_child_added = False
            for j, tac in enumerate(tactic_candidates):
                node_id = (tuple(tactic_path), tac)
                try:
                    next_state = dojo.run_tac(cur_state, tac)
                    success = not isinstance(next_state, LeanError)
                    # Always log proof state for child node if new and state returned
                    if success and next_state is not None:
                        pp_child = getattr(next_state, "pp", str(next_state))
                        dag_node_states[tuple(tactic_path) + (tac,)] = {
                            "pp": pp_child,
                        }
                    dag_results.append({
                        "depth": cur_depth,
                        "tactic_path": list(tactic_path),
                        "tactic": tac,
                        "success": success,
                        "cur_state_pp": cur_state.pp if hasattr(cur_state, "pp") else str(cur_state),
                        "next_state_pp": getattr(next_state, "pp", str(next_state)) if next_state else None,
                    })
                    dag_edges.append({
                        "from": list(tactic_path),
                        "tactic": tac,
                        "to": list(tactic_path) + [tac] if success else None,
                        "success": success,
                    })
                    if success:
                        num_success += 1
                        success_for_this_node += 1
                        # Use new success criterion for proof finished
                        if is_state_proof_complete(next_state):
                            print(f"      !!! Complete proof found with tactic: {tac} (path: {tactic_path + [tac]})")
                            found_complete_proof = True
                            complete_proof_for_this_node = True
                        if not complete_proof_for_this_node and not first_success_child_added:
                            new_states_to_explore.append((next_state, tactic_path + [tac]))
                            first_success_child_added = True
                        # Best-first: As soon as we hit one success, stop considering further tactics for this node.
                        if first_success_child_added:
                            break
                    else:
                        num_failed += 1
                        fail_for_this_node += 1
                except Exception as e:
                    dag_results.append({
                        "depth": cur_depth,
                        "tactic_path": list(tactic_path),
                        "tactic": tac,
                        "success": False,
                        "cur_state_pp": cur_state.pp if hasattr(cur_state, "pp") else str(cur_state),
                        "next_state_pp": None,
                        "error": repr(e),
                    })
                    dag_edges.append({
                        "from": list(tactic_path),
                        "tactic": tac,
                        "to": None,
                        "success": False,
                    })
                    num_failed += 1
                    fail_for_this_node += 1
                # For best_first, success or not, break after first success as above.
        else:
            successful_children = 0
            for j, tac in enumerate(tactic_candidates):
                if successful_children >= MAX_CHILDREN_PER_NODE:
                    break
                node_id = (tuple(tactic_path), tac)
                try:
                    next_state = dojo.run_tac(cur_state, tac)
                    success = not isinstance(next_state, LeanError)
                    # Always log proof state for child node if new and state returned
                    if success and next_state is not None:
                        pp_child = getattr(next_state, "pp", str(next_state))
                        dag_node_states[tuple(tactic_path) + (tac,)] = {
                            "pp": pp_child,
                        }
                    dag_results.append({
                        "depth": cur_depth,
                        "tactic_path": list(tactic_path),
                        "tactic": tac,
                        "success": success,
                        "cur_state_pp": cur_state.pp if hasattr(cur_state, "pp") else str(cur_state),
                        "next_state_pp": getattr(next_state, "pp", str(next_state)) if next_state else None,
                    })
                    dag_edges.append({
                        "from": list(tactic_path),
                        "tactic": tac,
                        "to": list(tactic_path) + [tac] if success else None,
                        "success": success,
                    })
                    if success:
                        num_success += 1
                        success_for_this_node += 1
                        if is_state_proof_complete(next_state):
                            print(f"      !!! Complete proof found with tactic: {tac} (path: {tactic_path + [tac]})")
                            found_complete_proof = True
                            complete_proof_for_this_node = True
                        if not complete_proof_for_this_node and successful_children < MAX_CHILDREN_PER_NODE:
                            new_states_to_explore.append((next_state, tactic_path + [tac]))
                            successful_children += 1
                    else:
                        num_failed += 1
                        fail_for_this_node += 1
                except Exception as e:
                    dag_results.append({
                        "depth": cur_depth,
                        "tactic_path": list(tactic_path),
                        "tactic": tac,
                        "success": False,
                        "cur_state_pp": cur_state.pp if hasattr(cur_state, "pp") else str(cur_state),
                        "next_state_pp": None,
                        "error": repr(e),
                    })
                    dag_edges.append({
                        "from": list(tactic_path),
                        "tactic": tac,
                        "to": None,
                        "success": False,
                    })
                    num_failed += 1
                    fail_for_this_node += 1

        total_this_node = success_for_this_node + fail_for_this_node
        if total_this_node > 0:
            success_pct = 100 * success_for_this_node / total_this_node
            fail_pct = 100 * fail_for_this_node / total_this_node
        else:
            success_pct = 0.0
            fail_pct = 0.0
        print(f"      Node {i+1}: Success %: {success_pct:.1f} | Fail %: {fail_pct:.1f} "
              f"(S/F/Total: {success_for_this_node}/{fail_for_this_node}/{total_this_node})")
        node_stats_this_level.append((i+1, success_pct, fail_pct, total_this_node))

    print(f"  Finished depth {cur_depth}, moving to next level. (success: {num_success}, failed: {num_failed})\n")
    cur_depth += 1
    states_to_explore = new_states_to_explore  # For the next level

    if found_complete_proof:
        print(f"Proof complete for at least one state at depth {cur_depth-1}. Stopping search.")
        break
else:
    if not found_complete_proof:
        print("Proof not found up to max depth or all states exhausted.")


In [34]:

# Print non-successful percentages in top X tactics for each depth (level summary)
print("\nSummary of non-successful and successful percentages across DAG levels:")
max_seen_depth = max((r["depth"] for r in dag_results), default=-1)
for depth in range(max_seen_depth + 1):
    total = sum(1 for r in dag_results if r["depth"] == depth)
    failures = sum(1 for r in dag_results if r["depth"] == depth and not r["success"])
    successes = sum(1 for r in dag_results if r["depth"] == depth and r["success"])
    if total > 0:
        non_success_pct = 100 * failures / total
        success_pct = 100 * successes / total
    else:
        non_success_pct = float('nan')
        success_pct = float('nan')
    print(f"Depth {depth}: Success %: {success_pct:.1f}, Failure %: {non_success_pct:.1f}, total tried: {total}")

# Save DAG with all associated info to disk
dag_info = {
    "nodes": None,  # Not explicitly listing, but can be reconstructed from edges/results
    "edges": dag_edges,
    "results": dag_results,
    "X": X,
    "N_LEVELS": cur_depth if found_complete_proof else cur_depth,  # true layers explored
    "MAX_CHILDREN_PER_NODE": MAX_CHILDREN_PER_NODE,
    "best_first": best_first,
}

thm_name = getattr(theorem, 'full_name', 'unknown_theorem').replace(".", "_")
dag_fname = f"{thm_name}_DAG.pkl"
with open(dag_fname, "wb") as f:
    pickle.dump(dag_info, f)
print(f"\nSaved DAG and all info to {dag_fname}\n")

In [37]:
# Extract unique nodes from edges
all_nodes = set()
for edge in dag_edges:
    source = edge.get('source', edge.get('from'))
    target = edge.get('target', edge.get('to'))
    
    # Convert to tuple if it's a list (to make it hashable)
    if isinstance(source, list):
        source = tuple(source)
    if isinstance(target, list):
        target = tuple(target)
    
    all_nodes.add(source)
    all_nodes.add(target)

# Convert to list and sort for consistent output
nodes_list = sorted(list(all_nodes), key=str)

print(f"\nTotal nodes in DAG: {len(nodes_list)}")
print(f"Total edges: {len(dag_edges)}")
print(f"\nFirst 10 nodes:")
print("=" * 60)

for i, node_id in enumerate(nodes_list[:10], 1):
    # Convert back to list for comparison if needed
    node_id_for_compare = list(node_id) if isinstance(node_id, tuple) else node_id
    
    # Find edges where this node is the source
    outgoing = []
    for e in dag_edges:
        src = e.get('source', e.get('from'))
        if (isinstance(src, list) and isinstance(node_id, tuple) and list(node_id) == src) or src == node_id:
            outgoing.append(e)
    
    # Find edges where this node is the target
    incoming = []
    for e in dag_edges:
        tgt = e.get('target', e.get('to'))
        if (isinstance(tgt, list) and isinstance(node_id, tuple) and list(node_id) == tgt) or tgt == node_id:
            incoming.append(e)
    
    # Find results for this node
    node_results = []
    for r in dag_results:
        result_node_id = r.get('node_id', r.get('state_id'))
        if result_node_id == node_id or (isinstance(result_node_id, list) and isinstance(node_id, tuple) and result_node_id == list(node_id)):
            node_results.append(r)
    
    print(f"\nNode {i}: ID={node_id}")
    print(f"  Incoming edges: {len(incoming)}")
    print(f"  Outgoing edges: {len(outgoing)}")
    if node_results:
        result = node_results[0]
        print(f"  Depth: {result.get('depth', 'N/A')}")
        print(f"  Success: {result.get('success', False)}")
        if 'tactic' in result:
            print(f"  Tactic: {result['tactic'][:60]}...")
    if outgoing:
        child_ids = [e.get('target', e.get('to')) for e in outgoing[:3]]
        print(f"  Children: {child_ids}")

# Print nodes at each depth level
print("\n" + "=" * 60)
print("Sample nodes by depth:")
print("=" * 60)

max_depth = max((r.get("depth", 0) for r in dag_results), default=0)
for depth in range(min(3, max_depth + 1)):  # Show first 3 depths
    depth_nodes = [r for r in dag_results if r.get("depth") == depth]
    print(f"\nDepth {depth} (showing first 3 nodes):")
    for i, result in enumerate(depth_nodes[:3], 1):
        node_id = result.get('node_id', result.get('state_id', 'unknown'))
        print(f"  {i}. Node ID: {node_id}")
        print(f"     Success: {result.get('success', False)}")
        if 'tactic' in result:
            print(f"     Tactic: {result['tactic'][:50]}...")
        if 'state_pp' in result:
            state_preview = result['state_pp'][:100].replace('\n', ' ')
            print(f"     State: {state_preview}...")

# tactic+premises fix?
need premises corpus


In [90]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("kaiyuy/leandojo-lean4-retriever-tacgen-byt5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("kaiyuy/leandojo-lean4-retriever-tacgen-byt5-small")

state = "n : ℕ\n⊢ gcd n n = n"
retrieved_premises = [
  "def <a>Nat.gcd</a> : Nat → Nat → Nat\n| 0        y := y\n| (succ x) y := have y % succ x < succ x, from mod_lt _ $ succ_pos _,\n                gcd (y % succ x) (succ x)",
  "@[simp] theorem <a>Nat.mod_self</a> (n : Nat) : n % n = 0",
]
input = "\n\n".join(retrieved_premises + [state])
print("------ INPUT ------\n", input)
tokenized_input = tokenizer(input, return_tensors="pt", max_length=2300, truncation=True)

# Generate a single tactic.
tactic_ids = model.generate(tokenized_input.input_ids, max_length=1024)
tactic = tokenizer.decode(tactic_ids[0], skip_special_tokens=True)
print("\n------ OUTPUT ------")
print(tactic, end="\n\n")

# Generate multiple tactics via beam search.
tactic_candidates_ids = model.generate(
    tokenized_input.input_ids,
    max_length=1024,
    num_beams=4,
    length_penalty=0.0,
    do_sample=False,
    num_return_sequences=4,
    early_stopping=False,
)
tactic_candidates = tokenizer.batch_decode(
    tactic_candidates_ids, skip_special_tokens=True
)
for tac in tactic_candidates:
    print(tac)

In [91]:
tokenized_state = tokenizer(thm_pointer['state_before'], return_tensors="pt")

# Generate a single tactic.
tactic_ids = model.generate(tokenized_state.input_ids, max_length=1024)
tactic = tokenizer.decode(tactic_ids[0], skip_special_tokens=True)
print(tactic, end="\n\n")

state_1 = dojo.run_tac(state, tactic)

print(state_1)


In [ ]:

# # state = "n : ℕ\n⊢ gcd n n = n"
# # state=edges[0]['state_before']


# # Generate multiple tactics via beam search.
# tactic_candidates_ids = model.generate(
#     tokenized_state.input_ids,
#     max_length=1024,
#     num_beams=4,
#     length_penalty=0.0,
#     do_sample=False,
#     num_return_sequences=4,
#     early_stopping=False,
# )
# tactic_candidates = tokenizer.batch_decode(
#     tactic_candidates_ids, skip_special_tokens=True
# )
# for tac in tactic_candidates:
#     print(tac)